# 01 - Dataset1 Classification Final Test

Bu notebook, Dataset1 için seçili sınıflandırma modellerini test split üzerinde değerlendirir.

Amaç, model eğitimi yapmadan Linear SVM, RBF SVM ve Random Forest ailelerini aynı test verisi üzerinde karşılaştırmaktır.


## 1. Kütüphaneler

Bu bölümde model dosyalarını yüklemek, parquet feature tablolarını okumak, test metriklerini hesaplamak ve görselleri oluşturmak için kullanılan kütüphaneler yüklenir.


In [ ]:
from pathlib import Path
from datetime import datetime
import re
import traceback
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

warnings.filterwarnings("ignore")

try:
    import pyarrow  # noqa: F401
    PARQUET_ENGINE = "pyarrow"
    print(f"[OK] pyarrow available: {pyarrow.__version__}")
except ImportError as exc:
    raise ImportError(
        "pyarrow is required for reading Stage 03 parquet feature files. "
        "Install it with: conda install -c conda-forge pyarrow"
    ) from exc


## 2. Ayarlar

Dataset bilgileri, test split, beklenen model aileleri ve overwrite davranışı tanımlanır.


In [ ]:
DATASET_NAME = "dataset1_varroa"
DATASET_DISPLAY_NAME = "Dataset1 Varroa"
DATASET_SHORT = "d1"

STAGE_NAME = "07_classification_final_test"
NOTEBOOK_NAME = "01_dataset1_classification_final_test"

RANDOM_STATE = 42

# Bu notebook yalnızca test split üzerinde değerlendirme yapar.
EVALUATION_SPLIT = "test"
PRIMARY_RANKING_METRIC = "test_f1"

# Beklenen model ailesi ve kombinasyon sayıları
EXPECTED_SELECTED_COMBINATION_COUNT = 9
EXPECTED_MODEL_FAMILIES = ["linear_svm", "rbf_svm", "random_forest"]
EXPECTED_MODEL_COUNT = EXPECTED_SELECTED_COMBINATION_COUNT * len(EXPECTED_MODEL_FAMILIES)

# Model ve feature dosyaları bu notebookta yazılmaz.
OVERWRITE_CLASSIFICATION_RESULTS = False
OVERWRITE_TABLES = False
OVERWRITE_FIGURES = False

DISPLAY_PREVIEW_ROWS = 10
REPORT_MAX_TABLE_ROWS = 50

NOTEBOOK_STARTED_AT = datetime.now()

print(f"[START] {NOTEBOOK_NAME} at {NOTEBOOK_STARTED_AT:%Y-%m-%d %H:%M:%S}")
print(f"[INFO] Dataset: {DATASET_NAME}")
print(f"[INFO] Evaluation split: {EVALUATION_SPLIT}")
print(f"[INFO] Expected selected combinations: {EXPECTED_SELECTED_COMBINATION_COUNT}")
print(f"[INFO] Expected model families: {EXPECTED_MODEL_FAMILIES}")
print(f"[INFO] Expected model count: {EXPECTED_MODEL_COUNT}")


## 3. Yardımcı Fonksiyonlar ve Dosya Yolları

Proje kökü, feature klasörü, seçili model havuzu ve bu notebooka ait tablo/görsel çıktı klasörleri hazırlanır.


In [ ]:
def find_project_root(start_path=None):
    if start_path is None:
        start_path = Path.cwd()
    start_path = Path(start_path).resolve()

    candidates = [start_path] + list(start_path.parents)
    for candidate in candidates:
        if (candidate / "data").exists() and (candidate / "approaches").exists():
            return candidate

    raise FileNotFoundError(
        "Proje kökü bulunamadı. Notebook'u proje klasörü içinde çalıştırdığından emin ol."
    )


def normalize_path_from_csv(path_value, project_root):
    if pd.isna(path_value) or str(path_value).strip() == "":
        return None

    raw = str(path_value).strip()
    path = Path(raw)

    if path.exists():
        return path

    normalized = raw.replace(chr(92), "/")
    for anchor in ["data/features/", "outputs/models/", "approaches/"]:
        if anchor in normalized:
            candidate = project_root / anchor.rstrip("/") / normalized.split(anchor, 1)[1]
            if candidate.exists():
                return candidate

    if raw.endswith(".parquet"):
        candidate = project_root / "data" / "features" / DATASET_NAME / Path(raw).name
        if candidate.exists():
            return candidate

    if raw.endswith(".joblib"):
        selected_pool = project_root / "outputs" / "models" / "selected_classification_model_pool" / DATASET_NAME
        for algorithm_dir in selected_pool.glob("*"):
            candidate = algorithm_dir / Path(raw).name
            if candidate.exists():
                return candidate

    return project_root / normalized


def relative_path(path):
    path = Path(path)

    try:
        return path.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()
    except Exception:
        return path.as_posix()


PROJECT_ROOT = find_project_root()
APPROACH_DIR = PROJECT_ROOT / "approaches" / "traditional_ml"

FEATURES_DIR = PROJECT_ROOT / "data" / "features" / DATASET_NAME
SELECTED_MODEL_POOL_DIR = PROJECT_ROOT / "outputs" / "models" / "selected_classification_model_pool" / DATASET_NAME

STAGE07_SELECTED_CANDIDATES_PATH = (
    APPROACH_DIR
    / "inputs"
    / STAGE_NAME
    / "selected_linear_candidates_all.csv"
)

OUTPUT_DIR = APPROACH_DIR / "outputs" / STAGE_NAME / NOTEBOOK_NAME
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"

for directory in [TABLES_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

required_paths = {
    "FEATURES_DIR": FEATURES_DIR,
    "SELECTED_MODEL_POOL_DIR": SELECTED_MODEL_POOL_DIR,
    "STAGE07_SELECTED_CANDIDATES_PATH": STAGE07_SELECTED_CANDIDATES_PATH,
}

for name, path in required_paths.items():
    if not Path(path).exists():
        raise FileNotFoundError(f"{name} not found: {path}")

print(f"[OK] Project root: {PROJECT_ROOT}")
print(f"[OK] Features dir: {FEATURES_DIR}")
print(f"[OK] Selected model pool dir: {SELECTED_MODEL_POOL_DIR}")
print(f"[OK] Selected candidates: {STAGE07_SELECTED_CANDIDATES_PATH}")
print(f"[OK] Output dir: {OUTPUT_DIR}")


## 4. Kayıt ve Çıktı Yardımcıları

Tablo ve görsel çıktıları tek yerden yönetilir. Mevcut dosya varsa overwrite ayarına göre korunur veya yeniden yazılır.


In [ ]:
def timestamp_now():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def log_output(message):
    print(message)


def log_section(title):
    print()
    print("=" * 80)
    print(title)
    print("=" * 80)


def log_dataframe(title, df, max_rows=REPORT_MAX_TABLE_ROWS):
    print()
    print(f"[DATAFRAME] {title}")
    print(f"Shape: {df.shape}")
    display(df.head(max_rows))


def log_figure(title, description, figure_path):
    figure_path = Path(figure_path)
    print(f"[FIGURE] {title}: {figure_path}")


def save_dataframe_csv(df, output_path, overwrite=False, index=False, note=""):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists() and not overwrite:
        print(f"[SKIP] Existing CSV kept: {output_path}")
        try:
            existing = pd.read_csv(output_path)
            return existing, "loaded_existing"
        except pd.errors.EmptyDataError:
            print(f"[WARNING] Existing CSV is empty; current dataframe kept in memory: {output_path}")
            return df, "kept_current_dataframe"

    if output_path.exists() and overwrite:
        print(f"[OVERWRITE] Replacing CSV: {output_path}")

    df.to_csv(output_path, index=index, encoding="utf-8-sig")
    print(f"[SAVE] CSV saved: {output_path}")
    return df, "created_or_overwritten"


def save_current_figure(figure_path, title, description="", overwrite=False, dpi=150):
    figure_path = Path(figure_path)
    figure_path.parent.mkdir(parents=True, exist_ok=True)

    if figure_path.exists() and not overwrite:
        plt.close()
        print(f"[SKIP] Existing figure kept: {figure_path}")
        log_figure(title, description, figure_path)
        return figure_path, "loaded_existing"

    if figure_path.exists() and overwrite:
        print(f"[OVERWRITE] Replacing figure: {figure_path}")

    plt.savefig(figure_path, dpi=dpi)
    plt.close()
    print(f"[SAVE] Figure saved: {figure_path}")
    log_figure(title, description, figure_path)
    return figure_path, "created_or_overwritten"


print("[OK] Kayıt ve çıktı yardımcıları hazır.")


## 5. Girdi Envanteri

Seçili aday tablosu, feature dosyaları ve model havuzu dosyaları kontrol edilir.


In [ ]:
log_section("01 INPUT INVENTORY")

selected_candidates_all_df = pd.read_csv(STAGE07_SELECTED_CANDIDATES_PATH)
dataset_candidates_df = (
    selected_candidates_all_df
    .query("dataset_name == @DATASET_NAME")
    .copy()
    .reset_index(drop=True)
)

if len(dataset_candidates_df) != EXPECTED_SELECTED_COMBINATION_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_SELECTED_COMBINATION_COUNT} selected combinations for {DATASET_NAME}, "
        f"but found {len(dataset_candidates_df)}."
    )

feature_files = sorted(FEATURES_DIR.glob("*.parquet"))
model_files = sorted(SELECTED_MODEL_POOL_DIR.glob("*/*.joblib"))

input_inventory_df = pd.DataFrame([
    {
        "input_name": "stage07_selected_candidates",
        "path": relative_path(STAGE07_SELECTED_CANDIDATES_PATH),
        "exists": STAGE07_SELECTED_CANDIDATES_PATH.exists(),
        "row_count": len(selected_candidates_all_df),
        "column_count": len(selected_candidates_all_df.columns),
    },
    {
        "input_name": "dataset_feature_files",
        "path": relative_path(FEATURES_DIR),
        "exists": FEATURES_DIR.exists(),
        "row_count": len(feature_files),
        "column_count": np.nan,
    },
    {
        "input_name": "selected_model_pool_files",
        "path": relative_path(SELECTED_MODEL_POOL_DIR),
        "exists": SELECTED_MODEL_POOL_DIR.exists(),
        "row_count": len(model_files),
        "column_count": np.nan,
    },
])

save_dataframe_csv(input_inventory_df, TABLES_DIR / "input_inventory.csv", overwrite=OVERWRITE_TABLES, note="Stage 07 input inventory")

log_dataframe("Input Inventory", input_inventory_df)
log_dataframe("Dataset Selected Candidate Mapping", dataset_candidates_df)


## 6. Mapping Kontrolü

Seçili aday kayıtlarının feature dosyalarıyla eşleştiği doğrulanır.


In [ ]:
log_section("02 STAGE 07 MAPPING VALIDATION")

required_mapping_columns = [
    "dataset_name",
    "dataset_short",
    "model_id",
    "patch_set",
    "patch_size",
    "ratio_variant",
    "feature_set",
    "feature_family",
    "selection_group",
    "selected_for_stage06",
    "source_feature_file_path",
]

missing_mapping_columns = [
    col for col in required_mapping_columns
    if col not in selected_candidates_all_df.columns
]

if missing_mapping_columns:
    raise ValueError(f"Stage 07 mapping file is missing required columns: {missing_mapping_columns}")

mapping_validation_rows = []

for _, row in dataset_candidates_df.iterrows():
    patch_set = row["patch_set"]
    feature_set = row["feature_set"]
    expected_feature_path = FEATURES_DIR / f"{patch_set}__{feature_set}.parquet"
    source_feature_path = normalize_path_from_csv(row.get("source_feature_file_path", ""), PROJECT_ROOT)

    mapping_validation_rows.append({
        "dataset_name": DATASET_NAME,
        "source_model_id": row["model_id"],
        "patch_set": patch_set,
        "feature_set": feature_set,
        "expected_feature_file_path": relative_path(expected_feature_path),
        "expected_feature_file_exists": expected_feature_path.exists(),
        "source_feature_file_path_resolved": relative_path(source_feature_path) if source_feature_path is not None else "",
        "source_feature_file_exists": bool(source_feature_path.exists()) if source_feature_path is not None else False,
        "mapping_status": "OK" if expected_feature_path.exists() else "MISSING_FEATURE_FILE",
    })

mapping_validation_df = pd.DataFrame(mapping_validation_rows)

if not mapping_validation_df["expected_feature_file_exists"].all():
    missing = mapping_validation_df.loc[
        ~mapping_validation_df["expected_feature_file_exists"],
        ["patch_set", "feature_set", "expected_feature_file_path"]
    ]
    display(missing)
    raise FileNotFoundError("One or more expected Stage 03 feature files are missing.")

save_dataframe_csv(mapping_validation_df, TABLES_DIR / "stage07_mapping_validation.csv", overwrite=OVERWRITE_TABLES, note="Stage 07 mapping validation")
log_dataframe("Stage 07 Mapping Validation", mapping_validation_df)


## 7. Model Havuzu Envanteri

Linear SVM, RBF SVM ve Random Forest model dosyalarının beklenen sayıda olup olmadığı kontrol edilir.


In [ ]:
log_section("03 SELECTED MODEL POOL INVENTORY")

algorithm_folder_map = {
    "linear_svm": "linear_svm",
    "rbf_svm": "rbf_svm",
    "random_forest": "random_forest",
}

model_inventory_rows = []

for algorithm_name, folder_name in algorithm_folder_map.items():
    algorithm_dir = SELECTED_MODEL_POOL_DIR / folder_name
    algorithm_files = sorted(algorithm_dir.glob("*.joblib")) if algorithm_dir.exists() else []
    model_inventory_rows.append({
        "dataset_name": DATASET_NAME,
        "algorithm_name": algorithm_name,
        "algorithm_dir": relative_path(algorithm_dir),
        "algorithm_dir_exists": algorithm_dir.exists(),
        "joblib_count": len(algorithm_files),
        "expected_count": EXPECTED_SELECTED_COMBINATION_COUNT,
        "status": "OK" if algorithm_dir.exists() and len(algorithm_files) == EXPECTED_SELECTED_COMBINATION_COUNT else "CHECK",
    })

model_pool_inventory_df = pd.DataFrame(model_inventory_rows)

if model_pool_inventory_df["joblib_count"].sum() != EXPECTED_MODEL_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_MODEL_COUNT} selected model artifacts for {DATASET_NAME}, "
        f"but found {model_pool_inventory_df['joblib_count'].sum()}."
    )

if not model_pool_inventory_df["status"].eq("OK").all():
    display(model_pool_inventory_df)
    raise ValueError("Selected model pool inventory check failed.")

save_dataframe_csv(model_pool_inventory_df, TABLES_DIR / "selected_model_pool_inventory.csv", overwrite=OVERWRITE_TABLES, note="selected model pool inventory")
log_dataframe("Selected Model Pool Inventory", model_pool_inventory_df)


## 8. Değerlendirme Planı

Her seçili aday ve model ailesi için test değerlendirme planı oluşturulur.


In [ ]:
log_section("04 EVALUATION PLAN CONSTRUCTION")

def find_selected_model_file(algorithm_name, patch_set, feature_set, source_linear_model_id=None):
    algorithm_dir = SELECTED_MODEL_POOL_DIR / algorithm_folder_map[algorithm_name]
    if not algorithm_dir.exists():
        raise FileNotFoundError(f"Algorithm folder not found: {algorithm_dir}")

    if algorithm_name == "linear_svm" and source_linear_model_id:
        exact_candidate = algorithm_dir / f"{source_linear_model_id}__{patch_set}__{feature_set}.joblib"
        if exact_candidate.exists():
            return exact_candidate

    pattern = f"*__{patch_set}__{feature_set}.joblib"
    matches = sorted(algorithm_dir.glob(pattern))

    if len(matches) == 1:
        return matches[0]

    if len(matches) == 0:
        raise FileNotFoundError(
            f"No model file found for algorithm={algorithm_name}, patch_set={patch_set}, feature_set={feature_set} "
            f"inside {algorithm_dir}"
        )

    raise ValueError(
        f"Multiple model files matched algorithm={algorithm_name}, patch_set={patch_set}, feature_set={feature_set}: "
        f"{[str(m) for m in matches]}"
    )


evaluation_plan_rows = []

for candidate_index, row in dataset_candidates_df.iterrows():
    patch_set = row["patch_set"]
    feature_set = row["feature_set"]
    feature_path = FEATURES_DIR / f"{patch_set}__{feature_set}.parquet"

    for algorithm_name in EXPECTED_MODEL_FAMILIES:
        model_path = find_selected_model_file(
            algorithm_name=algorithm_name,
            patch_set=patch_set,
            feature_set=feature_set,
            source_linear_model_id=row["model_id"],
        )

        evaluation_plan_rows.append({
            "dataset_name": DATASET_NAME,
            "dataset_short": DATASET_SHORT,
            "candidate_index": int(candidate_index + 1),
            "source_linear_model_id": row["model_id"],
            "algorithm_name": algorithm_name,
            "model_file_name": model_path.name,
            "model_file_path": relative_path(model_path),
            "patch_set": patch_set,
            "patch_size": row["patch_size"],
            "ratio_variant": row["ratio_variant"],
            "feature_set": feature_set,
            "feature_family": row["feature_family"],
            "selection_group": row["selection_group"],
            "feature_file_path": relative_path(feature_path),
            "evaluation_split": EVALUATION_SPLIT,
        })

classification_final_test_plan_df = pd.DataFrame(evaluation_plan_rows)

if len(classification_final_test_plan_df) != EXPECTED_MODEL_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_MODEL_COUNT} evaluation rows, found {len(classification_final_test_plan_df)}."
    )

missing_model_paths = classification_final_test_plan_df.loc[
    ~classification_final_test_plan_df["model_file_path"].map(lambda p: Path(p).exists())
]

missing_feature_paths = classification_final_test_plan_df.loc[
    ~classification_final_test_plan_df["feature_file_path"].map(lambda p: Path(p).exists())
]

if len(missing_model_paths) > 0:
    display(missing_model_paths)
    raise FileNotFoundError("Evaluation plan contains missing model paths.")

if len(missing_feature_paths) > 0:
    display(missing_feature_paths)
    raise FileNotFoundError("Evaluation plan contains missing feature paths.")

save_dataframe_csv(
    classification_final_test_plan_df,
    TABLES_DIR / "classification_final_test_plan.csv",
    overwrite=OVERWRITE_TABLES,
    note="Stage 07 classification final test evaluation plan",
)

log_dataframe("Classification Final Test Plan", classification_final_test_plan_df)


## 9. Feature ve Model Yükleme Yardımcıları

Model yükleme, feature kolon seçimi ve tek model test değerlendirme fonksiyonları tanımlanır.


In [ ]:
log_section("05 FEATURE AND MODEL LOADING HELPERS")

FEATURE_PREFIXES = ("hog8_", "hog16_", "hsv_", "lbp_")


def extract_estimator(loaded_object):
    """Return a fitted estimator or pipeline from common joblib payload formats."""
    if hasattr(loaded_object, "predict"):
        return loaded_object

    if isinstance(loaded_object, dict):
        candidate_keys = [
            "estimator",
            "pipeline",
            "model",
            "best_estimator",
            "best_estimator_",
            "trained_model",
        ]

        for key in candidate_keys:
            if key in loaded_object and hasattr(loaded_object[key], "predict"):
                return loaded_object[key]

    raise TypeError(
        "Loaded joblib object does not look like a fitted estimator or supported payload dictionary."
    )


def get_feature_columns(feature_df, estimator=None):
    """Select only handcrafted image feature columns and prevent metadata leakage."""
    if estimator is not None and hasattr(estimator, "feature_names_in_"):
        estimator_feature_names = list(estimator.feature_names_in_)
        missing = [col for col in estimator_feature_names if col not in feature_df.columns]
        if missing:
            raise ValueError(f"Feature file is missing estimator feature_names_in_ columns: {missing[:10]}")

        invalid_estimator_columns = [
            col for col in estimator_feature_names
            if not str(col).startswith(FEATURE_PREFIXES)
        ]

        if invalid_estimator_columns:
            raise ValueError(
                "Estimator feature_names_in_ contains columns outside approved feature prefixes. "
                f"Examples: {invalid_estimator_columns[:10]}"
            )

        return estimator_feature_names

    feature_columns = [
        col for col in feature_df.columns
        if str(col).startswith(FEATURE_PREFIXES)
    ]

    if not feature_columns:
        raise ValueError(
            "No image feature columns were found using FEATURE_PREFIXES. "
            f"Allowed prefixes: {FEATURE_PREFIXES}"
        )

    suspicious_keywords = [
        "label", "target", "class", "split", "patch_id", "patch_type",
        "negative", "positive", "source_bbox", "dataset", "path", "notes",
        "flags", "x1", "y1", "x2", "y2", "patch_size", "ratio_variant",
        "patch_set", "center", "width", "height", "area", "sampling",
    ]

    leaky_columns = [
        col for col in feature_columns
        if any(keyword in str(col).lower() for keyword in suspicious_keywords)
    ]

    if leaky_columns:
        raise ValueError(
            "Possible leakage columns were selected as features: "
            f"{leaky_columns}"
        )

    return feature_columns


def load_test_feature_data(feature_path, estimator=None):
    feature_df = pd.read_parquet(feature_path)

    if "split" not in feature_df.columns:
        raise ValueError(f"Feature file has no split column: {feature_path}")

    if "patch_label" not in feature_df.columns:
        raise ValueError(f"Feature file has no patch_label column: {feature_path}")

    test_df = feature_df[feature_df["split"].astype(str) == EVALUATION_SPLIT].copy()

    if test_df.empty:
        raise ValueError(f"No test rows found in feature file: {feature_path}")

    feature_columns = get_feature_columns(test_df, estimator=estimator)

    X_test = test_df[feature_columns]
    y_test = test_df["patch_label"].astype(int).to_numpy()

    return test_df, X_test, y_test, feature_columns


def compute_binary_metrics(y_true, y_pred):
    labels = [0, 1]
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    tn, fp, fn, tp = cm.ravel()

    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    return {
        "test_accuracy": accuracy_score(y_true, y_pred),
        "test_precision": precision_score(y_true, y_pred, zero_division=0),
        "test_recall": recall_score(y_true, y_pred, zero_division=0),
        "test_f1": f1_score(y_true, y_pred, zero_division=0),
        "test_specificity": specificity,
        "test_balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "test_tp": int(tp),
        "test_fp": int(fp),
        "test_tn": int(tn),
        "test_fn": int(fn),
    }


def evaluate_one_model(plan_row):
    model_path = Path(plan_row["model_file_path"])
    feature_path = Path(plan_row["feature_file_path"])

    loaded_model = joblib.load(model_path)
    estimator = extract_estimator(loaded_model)

    test_df, X_test, y_test, feature_columns = load_test_feature_data(feature_path, estimator=estimator)

    y_pred = estimator.predict(X_test).astype(int)
    metric_values = compute_binary_metrics(y_test, y_pred)

    base = {
        "dataset_name": DATASET_NAME,
        "dataset_short": DATASET_SHORT,
        "source_linear_model_id": plan_row["source_linear_model_id"],
        "algorithm_name": plan_row["algorithm_name"],
        "model_file_name": plan_row["model_file_name"],
        "patch_set": plan_row["patch_set"],
        "patch_size": plan_row["patch_size"],
        "ratio_variant": plan_row["ratio_variant"],
        "feature_set": plan_row["feature_set"],
        "feature_family": plan_row["feature_family"],
        "selection_group": plan_row["selection_group"],
        "evaluation_split": EVALUATION_SPLIT,
        "test_row_count": int(len(test_df)),
        "test_positive_count": int((y_test == 1).sum()),
        "test_negative_count": int((y_test == 0).sum()),
        "feature_column_count": int(len(feature_columns)),
        "model_file_path": relative_path(model_path),
        "feature_file_path": relative_path(feature_path),
    }

    metrics_row = {**base, **metric_values}

    confusion_rows = [
        {**base, "actual_label": 0, "predicted_label": 0, "count": int(metric_values["test_tn"]), "cell": "tn"},
        {**base, "actual_label": 0, "predicted_label": 1, "count": int(metric_values["test_fp"]), "cell": "fp"},
        {**base, "actual_label": 1, "predicted_label": 0, "count": int(metric_values["test_fn"]), "cell": "fn"},
        {**base, "actual_label": 1, "predicted_label": 1, "count": int(metric_values["test_tp"]), "cell": "tp"},
    ]

    return metrics_row, confusion_rows


print("[OK] Feature and model loading helpers are ready.")


## 10. Test Split Model Değerlendirmesi

Seçili modeller test split üzerinde değerlendirilir ve metrik tabloları oluşturulur.


In [ ]:
log_section("06 TEST SPLIT MODEL EVALUATION")

metric_rows = []
confusion_matrix_rows = []
evaluation_status_rows = []

for index, plan_row in classification_final_test_plan_df.iterrows():
    algorithm_name = plan_row["algorithm_name"]
    patch_set = plan_row["patch_set"]
    feature_set = plan_row["feature_set"]
    model_file_name = plan_row["model_file_name"]

    print(
        f"[RUN] {index + 1:03d}/{len(classification_final_test_plan_df):03d} — "
        f"{algorithm_name} — {patch_set} — {feature_set}"
    )

    try:
        metrics_row, confusion_rows = evaluate_one_model(plan_row)

        metric_rows.append(metrics_row)
        confusion_matrix_rows.extend(confusion_rows)

        evaluation_status_rows.append({
            "dataset_name": DATASET_NAME,
            "algorithm_name": algorithm_name,
            "model_file_name": model_file_name,
            "patch_set": patch_set,
            "feature_set": feature_set,
            "status": "OK",
            "error": "",
        })

    except Exception as exc:
        evaluation_status_rows.append({
            "dataset_name": DATASET_NAME,
            "algorithm_name": algorithm_name,
            "model_file_name": model_file_name,
            "patch_set": patch_set,
            "feature_set": feature_set,
            "status": "FAILED",
            "error": repr(exc),
        })
        print(f"[ERROR] Evaluation failed for {model_file_name}: {repr(exc)}")
        traceback.print_exc()
        raise

classification_final_test_metrics_df = pd.DataFrame(metric_rows)
classification_final_test_confusion_matrices_df = pd.DataFrame(confusion_matrix_rows)
evaluation_status_df = pd.DataFrame(evaluation_status_rows)

save_dataframe_csv(
    classification_final_test_metrics_df,
    TABLES_DIR / "classification_final_test_metrics.csv",
    overwrite=OVERWRITE_TABLES,
    note="Stage 07 test split model metrics",
)

save_dataframe_csv(
    classification_final_test_confusion_matrices_df,
    TABLES_DIR / "classification_final_test_confusion_matrices.csv",
    overwrite=OVERWRITE_TABLES,
    note="Stage 07 test split confusion matrices",
)

save_dataframe_csv(
    evaluation_status_df,
    TABLES_DIR / "evaluation_status.csv",
    overwrite=OVERWRITE_TABLES,
    note="Stage 07 model evaluation status",
)

log_dataframe("Evaluation Status", evaluation_status_df)
log_dataframe("Classification Final Test Metrics", classification_final_test_metrics_df)
log_dataframe("Classification Final Test Confusion Matrices", classification_final_test_confusion_matrices_df)


## 11. Sıralama ve Özet Tablolar

Test F1 metriğine göre model sıralaması ve algoritma/patch/feature özetleri oluşturulur.


In [ ]:
log_section("07 RANKING AND SUMMARY TABLES")

rank_sort_columns = [
    "test_f1",
    "test_recall",
    "test_precision",
    "test_specificity",
    "test_balanced_accuracy",
    "test_fp",
    "test_fn",
]
rank_sort_ascending = [False, False, False, False, False, True, True]

ranked_classification_final_test_models_df = (
    classification_final_test_metrics_df
    .sort_values(rank_sort_columns, ascending=rank_sort_ascending)
    .reset_index(drop=True)
)
ranked_classification_final_test_models_df.insert(
    0,
    "test_rank",
    np.arange(1, len(ranked_classification_final_test_models_df) + 1),
)

algorithm_summary_by_test_f1_df = (
    ranked_classification_final_test_models_df
    .groupby(["dataset_name", "algorithm_name"], as_index=False)
    .agg(
        model_count=("model_file_name", "count"),
        mean_test_f1=("test_f1", "mean"),
        median_test_f1=("test_f1", "median"),
        max_test_f1=("test_f1", "max"),
        mean_test_recall=("test_recall", "mean"),
        mean_test_precision=("test_precision", "mean"),
        mean_test_specificity=("test_specificity", "mean"),
        mean_test_balanced_accuracy=("test_balanced_accuracy", "mean"),
    )
    .sort_values(["max_test_f1", "mean_test_f1"], ascending=[False, False])
    .reset_index(drop=True)
)

patch_set_summary_by_test_f1_df = (
    ranked_classification_final_test_models_df
    .groupby(["dataset_name", "patch_set", "patch_size", "ratio_variant"], as_index=False)
    .agg(
        model_count=("model_file_name", "count"),
        mean_test_f1=("test_f1", "mean"),
        median_test_f1=("test_f1", "median"),
        max_test_f1=("test_f1", "max"),
        mean_test_recall=("test_recall", "mean"),
        mean_test_precision=("test_precision", "mean"),
    )
    .sort_values(["max_test_f1", "mean_test_f1"], ascending=[False, False])
    .reset_index(drop=True)
)

feature_set_summary_by_test_f1_df = (
    ranked_classification_final_test_models_df
    .groupby(["dataset_name", "feature_set", "feature_family"], as_index=False)
    .agg(
        model_count=("model_file_name", "count"),
        mean_test_f1=("test_f1", "mean"),
        median_test_f1=("test_f1", "median"),
        max_test_f1=("test_f1", "max"),
        mean_test_recall=("test_recall", "mean"),
        mean_test_precision=("test_precision", "mean"),
    )
    .sort_values(["max_test_f1", "mean_test_f1"], ascending=[False, False])
    .reset_index(drop=True)
)

# This is a performance-metric score summary, not a decision_function/probability diagnostic table.
classification_final_test_score_summary_df = (
    ranked_classification_final_test_models_df
    .groupby(["dataset_name", "algorithm_name"], as_index=False)
    .agg(
        model_count=("model_file_name", "count"),
        min_test_accuracy=("test_accuracy", "min"),
        mean_test_accuracy=("test_accuracy", "mean"),
        max_test_accuracy=("test_accuracy", "max"),
        min_test_precision=("test_precision", "min"),
        mean_test_precision=("test_precision", "mean"),
        max_test_precision=("test_precision", "max"),
        min_test_recall=("test_recall", "min"),
        mean_test_recall=("test_recall", "mean"),
        max_test_recall=("test_recall", "max"),
        min_test_f1=("test_f1", "min"),
        mean_test_f1=("test_f1", "mean"),
        max_test_f1=("test_f1", "max"),
    )
    .sort_values(["max_test_f1", "mean_test_f1"], ascending=[False, False])
    .reset_index(drop=True)
)

save_dataframe_csv(
    ranked_classification_final_test_models_df,
    TABLES_DIR / "ranked_classification_final_test_models.csv",
    overwrite=OVERWRITE_TABLES,
    note="Stage 07 models ranked by test_f1",
)

save_dataframe_csv(
    algorithm_summary_by_test_f1_df,
    TABLES_DIR / "algorithm_summary_by_test_f1.csv",
    overwrite=OVERWRITE_TABLES,
    note="Stage 07 algorithm summary by test_f1",
)

save_dataframe_csv(
    patch_set_summary_by_test_f1_df,
    TABLES_DIR / "patch_set_summary_by_test_f1.csv",
    overwrite=OVERWRITE_TABLES,
    note="Stage 07 patch set summary by test_f1",
)

save_dataframe_csv(
    feature_set_summary_by_test_f1_df,
    TABLES_DIR / "feature_set_summary_by_test_f1.csv",
    overwrite=OVERWRITE_TABLES,
    note="Stage 07 feature set summary by test_f1",
)

save_dataframe_csv(
    classification_final_test_score_summary_df,
    TABLES_DIR / "classification_final_test_score_summary.csv",
    overwrite=OVERWRITE_TABLES,
    note="Stage 07 prediction-based metric score summary",
)

log_dataframe("Ranked Classification Final Test Models", ranked_classification_final_test_models_df)
log_dataframe("Algorithm Summary by Test F1", algorithm_summary_by_test_f1_df)
log_dataframe("Patch Set Summary by Test F1", patch_set_summary_by_test_f1_df)
log_dataframe("Feature Set Summary by Test F1", feature_set_summary_by_test_f1_df)
log_dataframe("Classification Final Test Score Summary", classification_final_test_score_summary_df)


## 12. Test Görselleri

Test F1 sıralaması ve algoritma bazlı ortalama test F1 görselleri kaydedilir.


In [ ]:
log_section("08 VALIDATION FIGURES")

# Figure 1 — Test F1 by ranked model
fig1_path = FIGURES_DIR / "test_f1_by_ranked_model.png"
plot_df = ranked_classification_final_test_models_df.copy()
plt.figure(figsize=(12, 5))
plt.bar(np.arange(len(plot_df)) + 1, plot_df["test_f1"])
plt.xlabel("Test rank")
plt.ylabel("Test F1")
plt.title(f"{DATASET_DISPLAY_NAME} — Stage 07 Test F1 by Ranked Model")
plt.xticks(np.arange(len(plot_df)) + 1)
plt.tight_layout()
save_current_figure(fig1_path, "Test F1 by Ranked Model", "Selected models ranked by test F1.", overwrite=OVERWRITE_FIGURES)

# Figure 2 — Mean Test F1 by algorithm
fig2_path = FIGURES_DIR / "mean_test_f1_by_algorithm.png"
alg_plot_df = algorithm_summary_by_test_f1_df.sort_values("mean_test_f1", ascending=False).copy()
plt.figure(figsize=(8, 5))
plt.bar(alg_plot_df["algorithm_name"], alg_plot_df["mean_test_f1"])
plt.xlabel("Algorithm")
plt.ylabel("Mean Test F1")
plt.title(f"{DATASET_DISPLAY_NAME} — Mean Test F1 by Algorithm")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
save_current_figure(fig2_path, "Mean Test F1 by Algorithm", "Mean test F1 grouped by selected model family.", overwrite=OVERWRITE_FIGURES)

print("[OK] Validation figures created.")


## 13. Final Durum

Notebook sonunda test değerlendirme durumu kısa olarak özetlenir.


In [ ]:
log_section("09 FINAL DURUM")

notebook_finished_at = datetime.now()
evaluated_model_count = int(len(classification_final_test_metrics_df))
failed_evaluation_count = int((evaluation_status_df["status"] != "OK").sum())

notebook_status_df = pd.DataFrame([
    {
        "stage_name": STAGE_NAME,
        "notebook_name": NOTEBOOK_NAME,
        "dataset_name": DATASET_NAME,
        "evaluation_split": EVALUATION_SPLIT,
        "expected_model_count": EXPECTED_MODEL_COUNT,
        "evaluated_model_count": evaluated_model_count,
        "failed_evaluation_count": failed_evaluation_count,
        "best_model_file_name": ranked_classification_final_test_models_df.iloc[0]["model_file_name"] if evaluated_model_count else "",
        "best_algorithm_name": ranked_classification_final_test_models_df.iloc[0]["algorithm_name"] if evaluated_model_count else "",
        "best_patch_set": ranked_classification_final_test_models_df.iloc[0]["patch_set"] if evaluated_model_count else "",
        "best_feature_set": ranked_classification_final_test_models_df.iloc[0]["feature_set"] if evaluated_model_count else "",
        "best_test_f1": ranked_classification_final_test_models_df.iloc[0]["test_f1"] if evaluated_model_count else np.nan,
        "status": "READY_FOR_REVIEW" if evaluated_model_count == EXPECTED_MODEL_COUNT and failed_evaluation_count == 0 else "CHECK_REQUIRED",
        "started_at": NOTEBOOK_STARTED_AT.strftime("%Y-%m-%d %H:%M:%S"),
        "finished_at": notebook_finished_at.strftime("%Y-%m-%d %H:%M:%S"),
    }
])

notebook_status_df, _ = save_dataframe_csv(
    notebook_status_df,
    TABLES_DIR / "notebook_status.csv",
    overwrite=OVERWRITE_TABLES,
    note="notebook status",
)

log_dataframe("Notebook Status", notebook_status_df)
log_output(f"Status: {notebook_status_df.iloc[0]['status']}")
log_output(f"Evaluated model count: {evaluated_model_count}")
log_output(f"Failed evaluation count: {failed_evaluation_count}")
